<a href="https://colab.research.google.com/github/pySTEPS/ERAD-nowcasting-course-2026/blob/main/notebooks/exercise_notebooks/block_04_probabilistic_nowcasts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

All exercises can be found in this and the subsequent exercise blocks. The exercises should largely explain themselves and walk you through the different steps to prepare your data and create a nowcast. Note: These exercise notebooks are meant to help you to get started with the exercises. They are meant to be incomplete, so you will have to add some steps yourself (often indicated by the "..." in the code). The number of steps that have to be added by the participants progressively increases the further you get with the exercises. If you need to check your solutions, or if you are looking for the answers, see the [solutions](https://github.com/pySTEPS/ERAD-nowcasting-course-2026/tree/main/notebooks/solutions) notebooks.

# Nowcasting methods - part 2 probabilistic forecasts

In this notebook we show how to construct, visualize and apply verification metrics to a probablistic (ensemble) nowcast using pysteps.


## Load the data from the previous exercises

First we install pysteps, load and preprocess the example data by running the [helper_nowcasting_methods](https://github.com/pySTEPS/ERAD-nowcasting-course-2022/blob/hands-on-users/hands-on-session-users/notebooks/helper_nowcasting_methods.ipynb) notebook.

This helper notebook imports the radar data, dBR transforms it and determines the motion field with the Lucas-Kanada optical flow method (see [the notebook of block 3](https://github.com/pySTEPS/ERAD-nowcasting-course-2022/blob/hands-on-users/hands-on-session-users/notebooks/block_03_optical_flow_and_extrapolation.ipynb)). The precip data is split in a part for forecasting, called `precip_finite`, which is already dBR transformed and NaN values have been filled with a minimum value, and a part that will be used as observations (`precip_obs`) for model verification of the nowcasts.

The metadata corresponding to `precip_finite` is `metadata_dbr` and the metadata of `precip_obs` is `metadata`.

Finally the motion field variable is called `motion_field`. You can use these variables in these exercises.

In [ ]:
import os

# On Colab, mount Google Drive and switch to the notebook folder.
# Locally there is nothing to mount: Jupyter already runs in this folder.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # don't attempt to remount if the drive is already mounted
    if not os.path.exists("/content/mnt/MyDrive"):
        drive.mount("mnt")
    %cd '/content/mnt/MyDrive/Colab Notebooks/ERAD-nowcasting-course-2026/notebooks/exercise_notebooks/'

# run the data notebook to load the input dataset
%run helper_nowcasting_methods.ipynb

## Probabilistic nowcasts
In this part of the exercise, we are basically going to repeat the steps of the deterministic nowcast, but we will construct a probabilistic nowcast with 20 ensemble members and verify this nowcast accordingly.
If time allows, you can also try to make a LINDA-P nowcast.

The first step is to make a probablistic nowcast using the STEPS approach that is explained in [the STEPS nowcast gallery example](https://pysteps.readthedocs.io/en/latest/auto_examples/plot_steps_nowcast.html#stochastic-nowcast-with-steps). You can follow this example and adjust the code where necessary to make it work for our test case. The variable names of the already imported and pre-processed data have been mentioned above.

We are going to make an ensemble nowcast with 20 ensemble members and 12 lead times of 5 min (one hour in total). For a list of all options in the STEPS nowcast, see the [pysteps documentation](https://pysteps.readthedocs.io/en/latest/pysteps_reference/nowcasts.html#pysteps-nowcasts-steps).

In [ ]:
# Disable warnings
import warnings
warnings.filterwarnings("ignore")

from matplotlib import pyplot as plt
import numpy as np

from pysteps import nowcasts
from pysteps.postprocessing.ensemblestats import excprob
from pysteps.visualization import plot_precip_field

# Set nowcast parameters
n_ens_members = 20
n_leadtimes = 12
seed = 1234 # None gives a random seed number, but for reproducibility (i.e,
# every nowcast will give the same perturbations) we set it to a fixed number.

# The STEPS nowcast
nowcast_method = nowcasts.get_method("steps")
precip_forecast = nowcast_method(
    ...,
    ...,
    timesteps=n_leadtimes,
    n_ens_members=n_ens_members,
    n_cascade_levels=8,
    precip_thr=metadata_dbr["threshold"],
    kmperpixel=metadata_dbr["xpixelsize"]/1000.0,
    timestep=metadata_dbr["accutime"],
    noise_method="nonparametric",
    vel_pert_method="bps",
    probmatching_method="cdf",
    mask_method="incremental",
    seed=seed,
    num_workers=4,
)

# Back-transform the results from dBR to rain rates
...

### Visualize the result
We are going to visualize the observations, ensemble mean of the probabilistic nowcast, individual ensemble members of the nowcast and the forecast probability of exceeding a certain threshold (1 mm/h here). An example on how to do this is provided in [the STEPS nowcast gallery example](https://pysteps.readthedocs.io/en/latest/auto_examples/plot_steps_nowcast.html#stochastic-nowcast-with-steps).

In [ ]:
plt.figure(figsize=(16, 16))
# First plot the observations
for i, j in enumerate(range(2, 13, 3)):
    plt.subplot(4, 4, 1 + i).set_axis_off()
    plot_precip_field(...)
    plt.title(f"Observation at +{(j + 1) * 5} minutes")

# We'll plot the ensemble mean for four lead times
# First, obtain the ensemble mean from the forecast output
precip_forecast_mean = ...

plt.figure(figsize=(16, 16))
for i, j in enumerate(range(2, 13, 3)):
    plt.subplot(4, 4, 1 + i).set_axis_off()
    plot_precip_field(...)
    plt.title(f"Ensemble mean +{(j + 1) * 5} minutes")

# Then, plot some realizations
# Ensemble member 1
plt.figure(figsize=(16, 16))
for i, j in enumerate(range(2, 13, 3)):
    plt.subplot(4, 4, 1 + i).set_axis_off()
    plot_precip_field(...)
    plt.title(f"Ens. member 1 at +{(j + 1) * 5} minutes")

# Ensemble member 10
plt.figure(figsize=(16, 16))
for i, j in enumerate(range(2, 13, 3)):
    plt.subplot(4, 4, 1 + i).set_axis_off()
    plot_precip_field(...)
    plt.title(f"Ens. member 10 at +{(j + 1) * 5} minutes")

plt.show()



As you can see from the two shown members of the ensemble, the stochastic forecast maintains the same variance as in the observed rainfall field. Hence, it gives a less smoothed outcome than the ensemble mean and also preserves high-intensity rainfall cells. Keep this in mind for the verification part where we will use verification metrics that can take the entire ensemble into account.

In addition, we can also plot the probability of exceedance for a given threshold (1.0 mm/h in this example). See below:

In [ ]:
# Then plot the probability of exceeding 1 mm/h

plt.figure(figsize=(16, 10))
for i, j in enumerate(range(2, 13, 3)):
  # Compute exceedence probabilities for a 1.0 mm/h threshold
  P = excprob(precip_forecast[:, j, :, :], 1.0)
  plt.subplot(1, 4, 1 + i).set_axis_off()
  plot_precip_field(...)
  plt.title(f"Exceedance prob. at +{(j + 1) * 5} minutes")

plt.show()

### Ensemble forecast verification
Pysteps includes a number of verification metrics to help users to analyze the general characteristics of the nowcasts in terms of consistency and quality (or goodness). In contrast to the verification of the deterministic nowcast, we have a 20-member ensemble that we want to verify. As every member contains valuable information, it is better not to use the deterministic verification metrics on the ensemble mean, but to use a metric that can take the entire ensemble into account.

Therefore, we will focus on the CRPS (continuous ranked probability score), which you can see as the mean absolute error of the ensemble. It compares the cdf of the ensemble with the observed rainfall.

In addition, we will verify our probabilistic forecasts using the ROC curve, reliability diagrams, and rank histograms, as implemented in the [verification module](https://pysteps.readthedocs.io/en/latest/pysteps_reference/verification.html) of pysteps.

In [ ]:
from pysteps import verification
from pysteps.postprocessing import ensemblestats

# Determine the CRPS
CRPS = []
for lt in range(n_leadtimes):
    CRPS.append(verification.probscores.CRPS(...,...))

# Plot it
fig, ax1 = plt.subplots(figsize=(10, 4))

# Plot the CRPS for both lead times
ax1.plot(
    (np.arange(n_leadtimes)+1)*5,
    CRPS,
    color="blue",
    )

ax1.set_xlabel("Lead time (min)", fontsize=12)
ax1.set_ylabel(r"CRPS (mm h$^{-1}$)", fontsize=12)

ax1.set_title("CRPS")

plt.tight_layout()
plt.show()


The [examples gallery](https://pysteps.readthedocs.io/en/latest/auto_examples/plot_ensemble_verification.html#sphx-glr-auto-examples-plot-ensemble-verification-py) contains some ensemble verification examples that you could follow. For the subsequent verification metrics, we'll focus on a exceedance threshold of 1 mm/h. Try out some other thresholds and see how this influences the results.

In [ ]:
# We start with determining the exceedance probability in the forecast for a
# threshold of 1 mm/h for 1-h lead time (the last lead time in the forecast).
probability_forecast = ensemblestats.excprob(
    precip_forecast[:, -1, :, :],
    1.0,
    ignore_nan=True)

# ROC curve
roc = verification.ROC_curve_init(1.0, n_prob_thrs=9)
verification.ROC_curve_accum(
    ROC=...,
    P_f=...,
    X_o=...,
    )
fig, ax = plt.subplots()
verification.plot_ROC(roc, ax, opt_prob_thr=True)
ax.set_title("ROC curve (+%i min)" % (n_leadtimes * timestep))
plt.show()

In [ ]:
# Reliability diagram
reldiag = verification.reldiag_init(1.0)
verification.reldiag_accum(
    reldiag=...,
    P_f=...,
    X_o=...,
)
fig, ax = plt.subplots()
verification.plot_reldiag(reldiag, ax)
ax.set_title("Reliability diagram (+%i min)" % (n_leadtimes * timestep))
plt.show()

In [ ]:
# Rank histogram
rankhist = verification.rankhist_init(precip_forecast.shape[0], 1.0)
verification.rankhist_accum(
    ...,
    ...,
    ...
    )
fig, ax = plt.subplots()
verification.plot_rankhist(rankhist, ax)
ax.set_title("Rank histogram (+%i min)" % (n_leadtimes * timestep))
plt.show()

## LINDA-P Forecast
Lagrangian INtegro-Difference equation model with
Autoregression (LINDA) combines extrapolation, S-PROG, STEPS, ANVIL,
integro-difference equation (IDE) and cell tracking methods. It can produce
both deterministic and probabilistic nowcasts. LINDA is specifically designed
for nowcasting intense localized rainfall. For this purpose, it is expected to
give better forecast skill than S-PROG or STEPS.

In [ ]:
# Compute the probabilistic LINDA nowcast
nowcast_linda = nowcasts.linda.forecast(
    precip=precip_for_forecast[-3:, :, :],
    velocity=motion_field,
    timesteps=n_leadtimes,
    max_num_features=15,
    add_perturbations=True,
    n_ens_members=20,
    num_workers=4,
    seed=seed,
    measure_time=True,
    kmperpixel=metadata_dbr["xpixelsize"]/1000.0,
    timestep=metadata_dbr["accutime"],
)[0]

In [ ]:
# Plot two ensemble members of both nowcasts for the last lead time (+60 min)
# Feel free to try out some other lead times and some verification metrics.
fig = plt.figure(figsize = (12,12))
for i in range(2):
    ax = fig.add_subplot(2, 2, i + 1).set_axis_off()
    ax = plot_precip_field(
        nowcast_linda[i, -1, :, :],
        geodata=metadata,
        colorbar=False,
        axis="off",
        colorscale="STEPS-NL",
    )
    ax.set_title(f"LINDA Member {i+1}")

for i in range(2):
    ax = fig.add_subplot(2, 2, 3 + i).set_axis_off()
    ax = plot_precip_field(
        precip_forecast[i, -1, :, :],
        geodata=metadata,
        colorbar=False,
        axis="off",
        colorscale="STEPS-NL",
    )
    ax.set_title(f"STEPS Member {i+1}")